<a href="https://colab.research.google.com/github/akaleniuszka/03MIAR--Algoritmo-de-Optimizacion/blob/main/Proyecto/Proyecto_Alfredo_Kaleniuszka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Algoritmos de Optimización - Trabajo Práctico

Nombre y Apellidos: Alfredo Kaleniuszka  
URL GitHub: [Proyecto_Alfredo_Kaleniuszka.ipynb](https://github.com/akaleniuszka/03MIAR--Algoritmo-de-Optimizacion/blob/637ade26faa31a940e7457f7481a4bae91d3bb16/Proyecto/Proyecto_Alfredo_Kaleniuszka.ipynb)  

## Problema elegido: Organizar los horarios de partidos de La Liga

El objetivo del problema es asignar los partidos de **una jornada** de La Liga a los horarios disponibles, de forma que se maximice la audiencia total esperada.

El enunciado habla de organizar los horarios de los partidos de liga de cada jornada. Por tanto, se modela una jornada concreta compuesta por 10 partidos.

La audiencia de cada partido depende de:

- La categoría de los equipos enfrentados.
- El horario asignado.
- La penalización por coincidencia con otros partidos en el mismo horario.

Además, debe asignarse obligatoriamente al menos un partido el viernes y al menos un partido el lunes.

In [1]:
# Librerías utilizadas en el trabajo
from itertools import product
from functools import lru_cache
import random
import pandas as pd

## Datos del problema

Se definen los horarios disponibles, la audiencia base según la categoría de los equipos, los coeficientes de reducción por horario y las penalizaciones por coincidencia.

In [2]:
# Horarios disponibles para asignar los partidos de una jornada.
# Cada horario se representa como una tupla: (día, hora).
horarios = [
    ("Viernes", 20),
    ("Sabado", 12),
    ("Sabado", 16),
    ("Sabado", 18),
    ("Sabado", 20),
    ("Domingo", 12),
    ("Domingo", 16),
    ("Domingo", 18),
    ("Domingo", 20),
    ("Lunes", 20)
]

# Audiencia base en millones según las categorías de los equipos enfrentados.
# Se toma como referencia el horario de sábado a las 20h.
audiencia_base = {
    ("A", "A"): 2.0,
    ("A", "B"): 1.3,
    ("A", "C"): 1.0,
    ("B", "B"): 0.9,
    ("B", "C"): 0.75,
    ("C", "C"): 0.47
}

# Coeficientes de ponderación según el horario asignado.
# El sábado a las 20h y el domingo a las 20h tienen coeficiente 1.
coef_horario = {
    ("Viernes", 20): 0.4,
    ("Sabado", 12): 0.55,
    ("Sabado", 16): 0.7,
    ("Sabado", 18): 0.8,
    ("Sabado", 20): 1.0,
    ("Domingo", 12): 0.45,
    ("Domingo", 16): 0.75,
    ("Domingo", 18): 0.85,
    ("Domingo", 20): 1.0,
    ("Lunes", 20): 0.4
}

# Penalización por coincidencia.
# La clave indica cuántos partidos coinciden con el partido considerado.
# Por ejemplo, si hay 2 partidos en el mismo horario, cada uno tiene 1 coincidencia.
penalizacion_coincidencias = {
    0: 0.00,
    1: 0.25,
    2: 0.45,
    3: 0.60,
    4: 0.70,
    5: 0.75,
    6: 0.78,
    7: 0.80,
    8: 0.80
}

## Jornada de ejemplo

Se utiliza la jornada de ejemplo del enunciado. Cada partido se representa mediante:

`(nombre del partido, categoría equipo 1, categoría equipo 2)`

In [3]:
# Jornada de ejemplo formada por 10 partidos.
# Las categorías A, B y C representan el nivel de audiencia de los equipos.
partidos = [
    ("Celta - Real Madrid", "B", "A"),
    ("Valencia - Real Sociedad", "B", "A"),
    ("Mallorca - Eibar", "C", "C"),
    ("Athletic - Barcelona", "B", "A"),
    ("Leganes - Osasuna", "C", "C"),
    ("Villarreal - Granada", "B", "C"),
    ("Alaves - Levante", "B", "B"),
    ("Espanyol - Sevilla", "B", "B"),
    ("Betis - Valladolid", "B", "C"),
    ("Atletico - Getafe", "B", "B")
]

In [4]:
def obtener_audiencia_base(cat1, cat2):
    """
    Devuelve la audiencia base según las categorías de los dos equipos.

    Se ordenan las categorías para que, por ejemplo, B-A y A-B se traten igual.
    """
    categorias = tuple(sorted((cat1, cat2)))
    return audiencia_base[categorias]


def audiencia_partido_en_horario(partido, horario):
    """
    Calcula la audiencia de un partido en un horario concreto,
    sin aplicar todavía la penalización por coincidencia.
    """
    _, cat1, cat2 = partido

    base = obtener_audiencia_base(cat1, cat2)
    coeficiente = coef_horario[horario]

    return base * coeficiente

## Número de posibilidades

Sin tener en cuenta restricciones, cada uno de los 10 partidos puede asignarse a cualquiera de los 10 horarios disponibles.

Por tanto, hay:

`10^10`

posibles asignaciones.

Teniendo en cuenta la restricción de que debe haber al menos un partido el viernes y al menos un partido el lunes, se puede calcular mediante inclusión-exclusión:

`10^10 - 2 * 9^10 + 8^10`

In [5]:
# Número de partidos y horarios del problema
n_partidos = len(partidos)
n_horarios = len(horarios)

# Sin restricciones: cada partido puede ir a cualquiera de los horarios.
sin_restricciones = n_horarios ** n_partidos

# Con restricciones: se exige que exista al menos un partido el viernes
# y al menos un partido el lunes.
# Se aplica inclusión-exclusión.
con_restricciones = (
    n_horarios**n_partidos
    - 2 * (n_horarios - 1)**n_partidos
    + (n_horarios - 2)**n_partidos
)

print("Posibilidades sin restricciones:", sin_restricciones)
print("Posibilidades con restricciones:", con_restricciones)

Posibilidades sin restricciones: 10000000000
Posibilidades con restricciones: 4100173022


## Estructura de datos elegida

Una solución se representa como una lista o tupla de asignaciones.

Cada posición representa un partido y el valor almacenado representa el índice del horario asignado.

Por ejemplo:

`[0, 4, 2, 8, ...]`

significa que el primer partido se asigna al horario 0, el segundo al horario 4, el tercero al horario 2, etc.

Esta estructura es adecuada porque permite:

- Evaluar rápidamente la audiencia total.
- Contar cuántos partidos coinciden en cada horario.
- Comprobar si se cumplen las restricciones de viernes y lunes.

## Función objetivo

La función objetivo consiste en maximizar la audiencia total esperada de la jornada.

Para cada partido se calcula:

`audiencia base * coeficiente horario * factor de coincidencia`

La audiencia total será la suma de las audiencias de todos los partidos.

Por tanto, es un problema de **maximización**.

In [6]:
def obtener_penalizacion(coincidencias):
    """
    Devuelve la penalización asociada al número de coincidencias.

    Si apareciera un caso no incluido explícitamente en la tabla,
    se utiliza la mayor penalización disponible.
    """
    if coincidencias in penalizacion_coincidencias:
        return penalizacion_coincidencias[coincidencias]

    max_coincidencias = max(penalizacion_coincidencias)
    return penalizacion_coincidencias[max_coincidencias]


def evaluar_asignacion(asignacion, partidos, horarios):
    """
    Evalúa una asignación completa de partidos a horarios.

    Parámetros:
    - asignacion: lista donde cada posición representa un partido y su valor
      indica el índice del horario asignado.
    - partidos: lista de partidos de la jornada.
    - horarios: lista de horarios disponibles.

    Devuelve:
    - audiencia_total: suma total de audiencias.
    - detalle: información desglosada de cada partido.
    """
    audiencia_total = 0
    detalle = []

    # Recorremos cada horario para saber qué partidos han sido asignados a él.
    for i_horario, horario in enumerate(horarios):
        indices_partidos = [
            i for i, h in enumerate(asignacion)
            if h == i_horario
        ]

        cantidad = len(indices_partidos)

        # Si no hay partidos en este horario, pasamos al siguiente.
        if cantidad == 0:
            continue

        # Si hay k partidos en el mismo horario, cada partido tiene k-1 coincidencias.
        coincidencias = cantidad - 1
        penalizacion = obtener_penalizacion(coincidencias)
        factor_coincidencia = 1 - penalizacion

        # Calculamos la audiencia final de cada partido asignado a este horario.
        for i_partido in indices_partidos:
            partido = partidos[i_partido]

            base_ponderada = audiencia_partido_en_horario(partido, horario)
            audiencia_final = base_ponderada * factor_coincidencia

            audiencia_total += audiencia_final

            detalle.append({
                "Partido": partido[0],
                "Categorias": partido[1] + "-" + partido[2],
                "Horario": f"{horario[0]} {horario[1]}h",
                "Base ponderada": round(base_ponderada, 3),
                "Coincidencias": coincidencias,
                "Audiencia final": round(audiencia_final, 3)
            })

    return audiencia_total, detalle

## Algoritmo por fuerza bruta

El algoritmo de fuerza bruta genera todas las asignaciones posibles de partidos a horarios.

Después descarta las que no cumplen la restricción de tener al menos un partido el viernes y al menos un partido el lunes.

Finalmente, evalúa la audiencia total y conserva la mejor asignación encontrada.

Para el caso completo hay demasiadas posibilidades, por lo que se muestra el algoritmo y se prueba con un subconjunto reducido de partidos.

In [7]:
def cumple_restricciones(asignacion):
    """
    Comprueba si una asignación cumple las restricciones obligatorias:
    debe haber al menos un partido el viernes y al menos un partido el lunes.
    """
    viernes = horarios.index(("Viernes", 20))
    lunes = horarios.index(("Lunes", 20))

    return viernes in asignacion and lunes in asignacion


def fuerza_bruta(partidos, horarios):
    """
    Resuelve el problema por fuerza bruta.

    Genera todas las posibles asignaciones, evalúa cada una y se queda
    con la que obtiene mayor audiencia total.
    """
    mejor_asignacion = None
    mejor_audiencia = -1

    # product genera todas las combinaciones posibles de horarios para los partidos.
    for asignacion in product(range(len(horarios)), repeat=len(partidos)):

        # Se descartan las asignaciones que no cumplen viernes y lunes.
        if not cumple_restricciones(asignacion):
            continue

        audiencia, _ = evaluar_asignacion(asignacion, partidos, horarios)

        # Si la asignación actual mejora la mejor conocida, se actualiza.
        if audiencia > mejor_audiencia:
            mejor_audiencia = audiencia
            mejor_asignacion = asignacion

    return mejor_asignacion, mejor_audiencia

In [8]:
# Para no ejecutar 10^10 combinaciones, se prueba fuerza bruta
# con un subconjunto reducido de 5 partidos.
partidos_pequenos = partidos[:5]

mejor_asignacion_fb, mejor_audiencia_fb = fuerza_bruta(partidos_pequenos, horarios)

print("Mejor audiencia por fuerza bruta con 5 partidos:", round(mejor_audiencia_fb, 3))
print("Mejor asignación:", mejor_asignacion_fb)

Mejor audiencia por fuerza bruta con 5 partidos: 4.081
Mejor asignación: (4, 7, 0, 8, 9)


## Complejidad del algoritmo por fuerza bruta

Si hay `n` partidos y `h` horarios, cada partido puede asignarse a cualquiera de los horarios.

La complejidad es:

`O(h^n)`

En este caso:

`10^10 = 10.000.000.000`

Por tanto, la fuerza bruta no es práctica para resolver la jornada completa.

## Algoritmo mejorado: programación dinámica con subconjuntos

Para mejorar la fuerza bruta se utiliza programación dinámica con máscaras de bits.

La idea es procesar los horarios uno a uno y decidir qué subconjunto de partidos se asigna a cada horario.

Cada estado queda definido por:

- El horario que se está procesando.
- El conjunto de partidos pendientes de asignar.

Esta técnica evita recalcular muchas asignaciones parciales.

In [9]:
def audiencia_subconjunto(mask, horario_idx, partidos, horarios):
    """
    Calcula la audiencia que se obtiene al asignar un subconjunto de partidos
    a un horario concreto.

    El subconjunto se representa mediante una máscara de bits.
    """
    indices = [
        i for i in range(len(partidos))
        if mask & (1 << i)
    ]

    cantidad = len(indices)

    # Si no se asigna ningún partido a este horario, la audiencia es cero.
    if cantidad == 0:
        return 0

    # Se calcula la penalización por coincidencia.
    coincidencias = cantidad - 1
    penalizacion = obtener_penalizacion(coincidencias)
    factor_coincidencia = 1 - penalizacion

    horario = horarios[horario_idx]
    total = 0

    # Sumamos la audiencia ponderada de todos los partidos del subconjunto.
    for i in indices:
        total += audiencia_partido_en_horario(partidos[i], horario)

    # Aplicamos el factor de coincidencia a todo el subconjunto.
    return total * factor_coincidencia

In [10]:
def programacion_dinamica(partidos, horarios):
    """
    Resuelve el problema mediante programación dinámica con subconjuntos.

    En cada paso se decide qué subconjunto de partidos se asigna
    al horario actual.
    """
    n = len(partidos)
    h = len(horarios)

    # Máscara con todos los partidos pendientes.
    # Si n = 10, todos será 1111111111 en binario.
    todos = (1 << n) - 1

    viernes_idx = horarios.index(("Viernes", 20))
    lunes_idx = horarios.index(("Lunes", 20))

    @lru_cache(None)
    def dp(horario_idx, pendientes):
        """
        Devuelve la mejor audiencia posible desde el horario actual
        considerando los partidos que todavía están pendientes.
        """
        # Caso base: si ya se procesaron todos los horarios.
        if horario_idx == h:
            if pendientes == 0:
                return 0, []
            return -float("inf"), []

        mejor_valor = -float("inf")
        mejor_plan = None

        # Recorremos todos los subconjuntos posibles de partidos pendientes.
        submask = pendientes

        while True:
            # Viernes y lunes deben tener al menos un partido asignado.
            if horario_idx in [viernes_idx, lunes_idx] and submask == 0:
                pass
            else:
                # Audiencia obtenida al asignar este subconjunto al horario actual.
                valor_actual = audiencia_subconjunto(
                    submask,
                    horario_idx,
                    partidos,
                    horarios
                )

                # Se resuelve recursivamente el resto del problema.
                valor_futuro, plan_futuro = dp(
                    horario_idx + 1,
                    pendientes ^ submask
                )

                valor_total = valor_actual + valor_futuro

                # Guardamos la mejor decisión encontrada.
                if valor_total > mejor_valor:
                    mejor_valor = valor_total
                    mejor_plan = [submask] + plan_futuro

            # Cuando submask llega a 0, ya se han recorrido todos los subconjuntos.
            if submask == 0:
                break

            # Forma eficiente de generar el siguiente subconjunto.
            submask = (submask - 1) & pendientes

        return mejor_valor, mejor_plan

    mejor_valor, mejor_plan = dp(0, todos)

    # Convertimos el plan basado en máscaras a una asignación normal.
    asignacion = [None] * n

    for horario_idx, mask in enumerate(mejor_plan):
        for i in range(n):
            if mask & (1 << i):
                asignacion[i] = horario_idx

    return asignacion, mejor_valor

In [11]:
# Aplicamos el algoritmo mejorado a la jornada completa.
mejor_asignacion, mejor_audiencia = programacion_dinamica(partidos, horarios)

print("Mejor audiencia encontrada:", round(mejor_audiencia, 3), "millones")
print("Mejor asignación:", mejor_asignacion)

Mejor audiencia encontrada: 6.856 millones
Mejor asignación: [8, 7, 9, 4, 0, 5, 6, 3, 1, 2]


In [12]:
# Evaluamos la mejor asignación para obtener una tabla detallada.
audiencia_total, detalle = evaluar_asignacion(
    mejor_asignacion,
    partidos,
    horarios
)

df_resultado = pd.DataFrame(detalle)
df_resultado

,Partido,Categorias,Horario,Base ponderada,Coincidencias,Audiencia final
0,Leganes - Osasuna,C-C,Viernes 20h,0.188,0,0.188
1,Betis - Valladolid,B-C,Sabado 12h,0.413,0,0.413
2,Atletico - Getafe,B-B,Sabado 16h,0.630,0,0.630
3,Espanyol - Sevilla,B-B,Sabado 18h,0.720,0,0.720
4,Athletic - Barcelona,B-A,Sabado 20h,1.300,0,1.300
5,Villarreal - Granada,B-C,Domingo 12h,0.338,0,0.338
6,Alaves - Levante,B-B,Domingo 16h,0.675,0,0.675
7,Valencia - Real Sociedad,B-A,Domingo 18h,1.105,0,1.105
8,Celta - Real Madrid,B-A,Domingo 20h,1.300,0,1.300
9,Mallorca - Eibar,C-C,Lunes 20h,0.188,0,0.188


In [13]:
# Mostramos la audiencia total optimizada de la jornada.
print("Audiencia total optimizada:", round(audiencia_total, 3), "millones")

Audiencia total optimizada: 6.856 millones


## Complejidad del algoritmo mejorado

El algoritmo mejorado utiliza programación dinámica sobre subconjuntos.

Para cada horario se analizan subconjuntos de los partidos pendientes. La complejidad aproximada es:

`O(h * 3^n)`

Aunque sigue siendo una complejidad exponencial, mejora claramente frente a la fuerza bruta `O(h^n)`.

En este problema, con 10 partidos y 10 horarios, la programación dinámica permite resolver la jornada completa de forma exacta.

## Juego de datos aleatorio

Se genera una jornada aleatoria respetando la distribución indicada por el enunciado:

- 3 equipos de categoría A.
- 11 equipos de categoría B.
- 6 equipos de categoría C.

Después se emparejan los 20 equipos para formar 10 partidos.

In [14]:
def generar_jornada_aleatoria(seed=42):
    """
    Genera una jornada aleatoria de 10 partidos.

    Se respeta la distribución del enunciado:
    3 equipos A, 11 equipos B y 6 equipos C.
    """
    random.seed(seed)

    # Creamos la lista de equipos con sus categorías.
    equipos = (
        [(f"A{i+1}", "A") for i in range(3)] +
        [(f"B{i+1}", "B") for i in range(11)] +
        [(f"C{i+1}", "C") for i in range(6)]
    )

    # Mezclamos aleatoriamente los equipos.
    random.shuffle(equipos)

    jornada = []

    # Emparejamos los equipos de dos en dos para formar partidos.
    for i in range(0, len(equipos), 2):
        equipo1, cat1 = equipos[i]
        equipo2, cat2 = equipos[i + 1]

        jornada.append((
            f"{equipo1} - {equipo2}",
            cat1,
            cat2
        ))

    return jornada


partidos_aleatorios = generar_jornada_aleatoria()
partidos_aleatorios

[('C6 - B3', 'C', 'B'),
 ('C1 - B2', 'C', 'B'),
 ('B7 - B11', 'B', 'B'),
 ('C2 - C5', 'C', 'C'),
 ('B4 - B10', 'B', 'B'),
 ('C4 - B8', 'C', 'B'),
 ('A2 - B9', 'A', 'B'),
 ('A3 - C3', 'A', 'C'),
 ('B5 - B6', 'B', 'B'),
 ('A1 - B1', 'A', 'B')]

In [15]:
# Aplicamos el algoritmo mejorado a la jornada aleatoria.
mejor_asignacion_random, mejor_audiencia_random = programacion_dinamica(
    partidos_aleatorios,
    horarios
)

# Evaluamos la solución obtenida.
audiencia_random, detalle_random = evaluar_asignacion(
    mejor_asignacion_random,
    partidos_aleatorios,
    horarios
)

df_random = pd.DataFrame(detalle_random)
df_random

,Partido,Categorias,Horario,Base ponderada,Coincidencias,Audiencia final
0,C4 - B8,C-B,Viernes 20h,0.300,0,0.300
1,C1 - B2,C-B,Sabado 12h,0.413,0,0.413
2,B5 - B6,B-B,Sabado 16h,0.630,0,0.630
3,B4 - B10,B-B,Sabado 18h,0.720,0,0.720
4,A1 - B1,A-B,Sabado 20h,1.300,0,1.300
5,C6 - B3,C-B,Domingo 12h,0.338,0,0.338
6,B7 - B11,B-B,Domingo 16h,0.675,0,0.675
7,A3 - C3,A-C,Domingo 18h,0.850,0,0.850
8,A2 - B9,A-B,Domingo 20h,1.300,0,1.300
9,C2 - C5,C-C,Lunes 20h,0.188,0,0.188


In [16]:
# Mostramos la audiencia total de la jornada aleatoria optimizada.
print("Audiencia total de la jornada aleatoria:", round(audiencia_random, 3), "millones")

Audiencia total de la jornada aleatoria: 6.713 millones


## Referencias

- Material de la asignatura Algoritmos de Optimización, VIU.
- Enunciado del Trabajo Práctico: Organización de horarios de partidos de La Liga.
- Documentación oficial de Python: `itertools`, `functools`, `random` y `pandas`.

## Líneas futuras

El problema se ha resuelto para una jornada concreta, que es lo que plantea el enunciado.

Como líneas futuras, podría ampliarse el modelo para optimizar una temporada completa. En ese caso habría que considerar restricciones adicionales, como calendario de ida y vuelta, descansos, partidos locales y visitantes, preferencias televisivas, restricciones de seguridad, disponibilidad de estadios y reparto justo de horarios entre equipos.

También podrían estudiarse técnicas como algoritmos genéticos, búsqueda tabú, recocido simulado o programación lineal entera para abordar instancias más grandes.